#----------------------------------------------------------------------------

# Entrenamiento de Modelos Baseline - Predicción de Cancelaciones Hoteleras

#----------------------------------------------------------------------------

Este notebook corresponde al rol de Baseline Trainer (PB-10).

### El objetivo es entrenar al menos cinco modelos baseline con parametros por defecto usando el pipeline de preprocesamiento con cada modelo dentro de un Pipeline completo


In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [7]:
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# =========================
# 1. Cargar datos
# =========================
X_train = pd.read_csv("../data/processed/X_train_bal.csv")
y_train = pd.read_csv("../data/processed/y_train_bal.csv").values.ravel()

# =========================
# 2. Cargar preprocesador
# =========================
preproc_dict = joblib.load("../models/preprocessor.pkl")

# Extraer componentes
feature_builder = preproc_dict["feature_builder"]
column_transformer = preproc_dict["preprocessor"]

# =========================
# 3. Definir modelos baseline
# =========================
models = {
    "lr": LogisticRegression(random_state=42, max_iter=1000),
    "dt": DecisionTreeClassifier(random_state=42),
    "rf": RandomForestClassifier(random_state=42),
    "svm": SVC(probability=True, random_state=42),
    "knn": KNeighborsClassifier(),
    "gb": GradientBoostingClassifier(random_state=42)
}

# =========================
# 4. Entrenar + guardar pipelines
# =========================
for name, model in models.items():
    
    pipe = Pipeline([
        ("feature_builder", feature_builder),
        ("preprocessor", column_transformer),
        ("clf", model)
    ])
    
    print(f"Entrenando modelo: {name}...")
    pipe.fit(X_train, y_train)
    
    output_path = f"../models/baseline_{name}.pkl"
    joblib.dump(pipe, output_path)
    
    print(f"Modelo guardado en: {output_path}")

Entrenando modelo: lr...
Modelo guardado en: ../models/baseline_lr.pkl
Entrenando modelo: dt...
Modelo guardado en: ../models/baseline_dt.pkl
Entrenando modelo: rf...
Modelo guardado en: ../models/baseline_rf.pkl
Entrenando modelo: svm...
Modelo guardado en: ../models/baseline_svm.pkl
Entrenando modelo: knn...
Modelo guardado en: ../models/baseline_knn.pkl
Entrenando modelo: gb...
Modelo guardado en: ../models/baseline_gb.pkl


#----------------------------------------------------------------------------
# **Modelos Baseline Utilizados**
#----------------------------------------------------------------------------

## 1. **Logistic Regression**

Es un modelo de clasificación lineal que estima la probabilidad de pertenecer a una clase utilizando una función logística.

¿Cómo funciona? -> Transforma una combinación lineal de las variables de entrada en una probabilidad entre 0 y 1 mediante la función sigmoide.

¿Por qué se incluye? ->

    - Sirve como baseline interpretable
    - Permite entender la relación entre variables y la variable objetivo
    - Es eficiente y rápido de entrenar

## 2. **Decision Tree**

Es un modelo basado en reglas que divide los datos en forma de árbol para tomar decisiones.

¿Cómo funciona? ->

    - Divide los datos en nodos usando reglas tipo: “si variable ≤ valor → izquierda, si no → derecha”.
    - Busca maximizar la separación entre clases (criterios como Gini o Entropía)

¿Por qué se incluye? ->

        - Fácil de interpretar
        - Captura relaciones no lineales
        - No requiere escalamiento de variables

## 3. **Random Forest**

Un conjunto (ensamble) de múltiples árboles de decisión.

¿Cómo funciona? ->

  - Entrena muchos árboles con:
    - subconjuntos aleatorios de datos
    - subconjuntos de variables
  - La predicción final es el promedio o voto mayoritario

¿Por qué se incluye? ->

    - Reduce el sobreajuste de un árbol individual
    - Mejora la precisión
    - Robusto frente a ruido

## 4. **Support Vector Machine (SVM)**

Un modelo que busca encontrar el hiperplano que mejor separa las clases.

¿Cómo funciona? ->

    - Maximiza la distancia (margen) entre clases
    - Puede usar kernels para separar datos no lineales
    
¿Por qué se incluye? ->

    - Alto rendimiento en datasets complejos
    - Funciona bien en espacios de alta dimensión
    - Buen benchmark para clasificación

## 5. **K-Nearest Neighbors (KNN)**
Un modelo basado en instancias que clasifica según los vecinos más cercanos.
    
¿Cómo funciona? ->

    - Calcula la distancia entre observaciones
    - Asigna la clase más común entre los k vecinos más cercanos

¿Por qué se incluye? ->

    - Modelo simple y no paramétrico
    - No asume forma funcional de los datos
    - Útil como referencia base

## 6. **Gradient Boosting**

Un modelo de ensamble que construye árboles secuencialmente para corregir errores previos.

¿Cómo funciona? ->

    - Cada nuevo árbol aprende de los errores del anterior
    - Minimiza una función de pérdida mediante descenso de gradiente
    
¿Por qué se incluye? ->

    - Alto rendimiento predictivo
    - Captura relaciones complejas
    - Base de modelos más avanzados (como XGBoost, LightGBM)


## El uso de estos modelos permite:

    * Comparar enfoques lineales vs no lineales
    * Evaluar modelos:
        - simples vs complejos
        - individuales vs ensambles
    * Establecer una línea base robusta antes de optimizar hiperparámetros